# 00 SETUP: Google Drive, datasets, literature

Run this notebook **once** (re-running is safe: whatever already exists is skipped). No GPU needed (*Runtime > Change runtime type > CPU* is fine).

It does four things:

1. It mounts Google Drive. Allow access when asked.
2. It creates the project layout in `MyDrive/IR_Lexical_Gap/project/`. The folders `src/`, `results/`, `cache/`, `figures/`, `report/` and `phase2/03_experiments|04_improvements/...` mirror the local project.
3. It downloads the 8 BEIR test collections into `MyDrive/IR_Lexical_Gap/data/<dataset>/`: nfcorpus, scifact, fiqa, scidocs, arguana, trec-covid, quora and webis-touche2020. That is about 1.3 GB unzipped and takes about 5-10 min.
4. It downloads every open-access PDF from `04_literature/reading_list.md` into `MyDrive/IR_Lexical_Gap/04_literature/pdfs/<topic>/FirstAuthorYear_ShortTitle.pdf` and writes `pdfs/download_report.txt`.
   - Papers marked "DOI only" in the reading list are paywalled (ACM, Springer, Elsevier). Fetch those from the university network.
   - The two Springer demo papers and the author-hosted Robertson & Zaragoza PDF may fail. The report lists every failure; nothing aborts.

**Then run, in this order:**
1. `phase1_full_pipeline_drive` (T4 GPU). It runs `lexical_gap_full_pipeline` with its outputs in Drive.
2. `phase2_dense_runs` (T4)
3. `phase2_ce_rerank` (T4)
4. `phase2_analysis` (CPU)

See `01_RUN_ORDER.md` in the Drive root.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install sentence-transformers PyStemmer wordfreq pytrec-eval-terrier reportlab statsmodels requests
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True); print('packages ok')

The setup code is `02_code/phase2/setup_drive.py` on Drive. It is copied to the runtime and run from there; open it in Drive to read it.

In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/IR_Lexical_Gap/02_code/phase2/setup_drive.py', '/content/setup_drive.py')  # Drive copy of phase2/colab/setup_drive.py

In [ ]:
!python /content/setup_drive.py layout

### BEIR datasets -> `MyDrive/IR_Lexical_Gap/data/`

In [ ]:
!python /content/setup_drive.py data

### Literature PDFs -> `MyDrive/IR_Lexical_Gap/04_literature/pdfs/<topic>/`

In [ ]:
!python /content/setup_drive.py papers

In [ ]:
import os, shutil
# keep a copy of the setup script in the Drive project mirror
shutil.copy('/content/setup_drive.py', '/content/drive/MyDrive/IR_Lexical_Gap/project/phase2/colab/setup_drive.py')
for root, dirs, files in sorted(os.walk('/content/drive/MyDrive/IR_Lexical_Gap/04_literature/pdfs')):
    pdfs = [f for f in files if f.endswith('.pdf')]
    if pdfs:
        print(root.split('pdfs/')[-1] + '/', len(pdfs), 'PDFs')
for d in sorted(os.listdir('/content/drive/MyDrive/IR_Lexical_Gap/data')):
    p = '/content/drive/MyDrive/IR_Lexical_Gap/data/' + d + '/corpus.jsonl'
    if os.path.exists(p):
        print(f'data/{d}: {os.path.getsize(p) / 1e6:.0f} MB corpus')
print('\nSetup finished. Next: open 01_notebooks/phase1_full_pipeline_drive.ipynb (T4 GPU) and run all.')